# VQ-VAE-2 Latent Feature Extraction

Extract and visualize latent representations from all 3 hierarchical levels of the VQ-VAE-2 encoder.

The new model architecture includes a `content_proj` layer (Conv3d) that projects content-only channels
back to `hidden_channels` after the level-0 encoder, so deeper encoders receive a full-channel spatial map.
This notebook correctly accounts for that by:
- Building the model with `content_size` and `style_size` (required to reconstruct `content_proj`)
- Using `pool_only=True` to get pooled `(B, C)` vectors per level (memory-efficient)
- Separately extracting **content** vs **style** channel subsets at level 0 using the Gumbel mask
- Visualising T1 vs T2 separation and diagnosis clustering at each hierarchical level

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
CHECKPOINT_PATH = "/home/ng24/projects/multiview-crl/results/ADNI_registered/vqvae-normal-logging-4/vqvae_model.pt"
DATA_DIR = "/data/natalia/ADNI_registered"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 100  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — content_size / style_size are required to reconstruct content_proj
vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=settings["vqvae_hidden_channels"],
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=settings["vqvae_nb_levels"],
    embed_dim=settings["vqvae_embed_dim"],
    nb_entries=settings["vqvae_nb_entries"],
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
)

hidden_channels = settings["vqvae_hidden_channels"]
content_channels = vqvae_model.content_channels  # actual channel count (may differ due to rounding)
print(f"hidden_channels : {hidden_channels}")
print(f"content_channels: {content_channels}  (level-0 pooled vectors will have this many dims)")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Handle old checkpoints that used ".blocks." instead of ".stack."
new_state_dict = {k.replace(".blocks.", ".stack."): v for k, v in state_dict.items()}

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)
if missing:
    print(f"⚠ Missing keys : {missing}")
if unexpected:
    print(f"⚠ Unexpected keys: {unexpected}")
if not missing and not unexpected:
    print("✓ Checkpoint loaded cleanly")
elif not missing:
    print("✓ Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"✓ Model loaded from step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd
from monai.transforms import ResizeWithPadOrCropd, NormalizeIntensityd, ToTensord

# Map RESAMPLE_MODE to the spacing value used by utils.utils.transforms()
if RESAMPLE_MODE == "2mm":
    spacing = 2.0
elif RESAMPLE_MODE == "original":
    spacing = 1.0
else:
    raise ValueError(f"Unknown RESAMPLE_MODE: {RESAMPLE_MODE}")

# Re-use the project's standard transform builder, but drop the label key
# (this notebook only works with image pairs, no label tensor needed)
base_keys = ["image_t1", "image_t2"]
mask_keys = ["mask_t1", "mask_t2"]
all_keys = base_keys + mask_keys

if spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

pre = [
    LoadImaged(keys=base_keys),
    EnsureChannelFirstd(keys=base_keys, channel_dim="no_channel"),
    CreateBrainMaskd(keys=base_keys, mask_keys=mask_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
]

if spacing != 1.0:
    pre += [
        Spacingd(keys=base_keys, pixdim=(spacing, spacing, spacing), mode="bilinear"),
        Spacingd(keys=mask_keys, pixdim=(spacing, spacing, spacing), mode="nearest"),
    ]

val_transforms = Compose(
    pre
    + [
        ResizeWithPadOrCropd(keys=all_keys, spatial_size=spatial_size),
        NormalizeIntensityd(keys=base_keys, nonzero=True, channel_wise=True),
        ApplyBrainMaskd(keys=base_keys, mask_keys=mask_keys, threshold=0.5),
        ToTensord(keys=base_keys),
    ]
)
print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}")

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, C)` pooled vectors, one per level
  - **Level 0**: `content_channels` dims (the content-masked, projected feature)
  - **Level 1+**: `hidden_channels` dims (full channel map pooled)
- `estimated_content_indices`: the Gumbel-selected channel indices at level 0

For visualisation we also split level-0 pooled features into **content** and **style** subsets.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []

# We'll also collect the raw level-0 FULL pooled features (hidden_channels dims)
# by running the encoder manually once, so we can split content vs style.
all_full_level0 = []  # (B, hidden_channels)
all_content_indices = None  # shared across samples (mask is computed per-forward from mean logits)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            _, _, enc_features, est_content_idx, _, _ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
            )

        # enc_features is a list of (1, C) pooled tensors
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())  # (C,)

        # For level-0: also grab the FULL hidden_channels pooled features
        # by running just encoder[0] directly (before content projection path)
        with torch.no_grad():
            spatial_l0 = vqvae_module.encoders[0](img)  # (1, hidden_channels, D, H, W)
            full_pool = spatial_l0.mean(dim=[2, 3, 4])  # (1, hidden_channels)
        all_full_level0.append(full_pool.squeeze(0).cpu().float().numpy())

        # Record the content indices from the last forward pass
        if all_content_indices is None and est_content_idx is not None:
            all_content_indices = est_content_idx[0]  # list of ints

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_full_level0 = np.array(all_full_level0)
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print("\n✓ Feature extraction complete!")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")
print(f"  full level-0 (hidden_channels): {all_full_level0.shape}")

# Derive content / style subsets from the full level-0 features
all_style_indices = [i for i in range(hidden_channels) if i not in set(all_content_indices or [])]
print(f"  content indices ({len(all_content_indices)} ch): {all_content_indices[:8]}...")
print(f"  style indices   ({len(all_style_indices)} ch):   {all_style_indices[:8]}...")

## 4. Feature statistics

In [ ]:
print("=" * 65)
print("Feature Statistics by Level (pooled, before quantization)")
print("=" * 65)

t1_mask = all_modalities == "T1"
t2_mask = all_modalities == "T2"

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    t1_f = features[t1_mask]
    t2_f = features[t2_mask]
    paired_dist = np.linalg.norm(t1_f - t2_f, axis=1)
    print(f"\nLevel {level_idx}  (shape {features.shape}):")
    print(
        f"  mean={features.mean():.4f}  std={features.std():.4f}  "
        f"min={features.min():.4f}  max={features.max():.4f}"
    )
    print(f"  T1-T2 paired ℓ2 distance — mean: {paired_dist.mean():.4f}  std: {paired_dist.std():.4f}")

# Extra breakdown for level 0: content vs style channels
if all_content_indices is not None:
    print("\n--- Level 0 channel breakdown (full hidden_channels pool) ---")
    content_f = all_full_level0[:, all_content_indices]
    style_f = all_full_level0[:, all_style_indices] if all_style_indices else None
    ct1 = content_f[t1_mask]
    ct2 = content_f[t2_mask]
    print(
        f"  Content  ({len(all_content_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(ct1-ct2,axis=1).mean():.4f}"
    )
    if style_f is not None and len(style_f):
        st1 = style_f[t1_mask]
        st2 = style_f[t2_mask]
        print(
            f"  Style    ({len(all_style_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(st1-st2,axis=1).mean():.4f}"
        )
    print("  (Content dist should be smaller than style dist if disentanglement is working)")

## 5. PCA visualisation

In [ ]:
colors_by_label = plt.cm.tab10(np.linspace(0, 1, max(len(label_map), 3)))

fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]  # keep 2-D indexing

for level_idx in range(nb_levels):
    # For level 0, show only the content channels — these are what the contrastive
    # loss acts on.  Plotting all hidden_channels at level 0 mixes in style channels,
    # which encode T1/T2 contrast differences and will dominate the first PC,
    # making the plot cluster by modality regardless of disentanglement quality.
    # The full content+style breakdown for level 0 is in Section 7.
    if level_idx == 0 and all_content_indices is not None:
        features = all_full_level0[:, all_content_indices]
        level_label = f"Level 0 — content ({len(all_content_indices)} ch)"
    else:
        features = all_features[f"level_{level_idx}"]
        level_label = f"Level {level_idx}"

    pca = PCA(n_components=2)
    f2d = pca.fit_transform(features)
    ev = pca.explained_variance_ratio_

    # Row 0 — colour by diagnosis
    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"{level_label} — Diagnosis\n({ev.sum()*100:.1f}% var)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    # Row 1 — colour by modality
    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"{level_label} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — PCA\n"
    "(Level 0: content channels only; Levels 1+: full hidden_channels; all pooled pre-quantization)",
    y=1.02,
    fontsize=12,
)
plt.tight_layout()
plt.savefig("latent_features_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_pca.png")

## 6. t-SNE visualisation

In [ ]:
fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features) - 1))
    f2d = tsne.fit_transform(features)

    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"Level {level_idx} — Diagnosis")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"Level {level_idx} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — t-SNE (pooled, before quantization)",
    y=1.02,
    fontsize=13,
)
plt.tight_layout()
plt.savefig("latent_features_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_tsne.png")

## 7. Content vs Style channel visualisation (level 0)

The Gumbel mask at level 0 selects `content_channels` out of `hidden_channels`.
If the model has learned to disentangle content (shared across T1/T2) from style
(view-specific), the **content** PCA should show T1 and T2 mixed together, while
the **style** PCA should separate them clearly.

In [ ]:
if all_content_indices is not None and len(all_style_indices) > 0:
    content_feats = all_full_level0[:, all_content_indices]
    style_feats = all_full_level0[:, all_style_indices]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for row, (feats, title_sfx) in enumerate(
        [
            (content_feats, f"Content ({len(all_content_indices)} ch)"),
            (style_feats, f"Style   ({len(all_style_indices)} ch)"),
        ]
    ):
        pca = PCA(n_components=2)
        f2d = pca.fit_transform(feats)
        ev = pca.explained_variance_ratio_

        # Col 0 — by diagnosis
        ax = axes[row, 0]
        for lab_idx, lab_name in label_names.items():
            m = all_labels == lab_idx
            ax.scatter(
                f2d[m, 0],
                f2d[m, 1],
                c=[colors_by_label[lab_idx]],
                label=lab_name,
                alpha=0.65,
                s=18,
            )
        ax.set_title(f"Level-0 {title_sfx}\nBy Diagnosis ({ev.sum()*100:.1f}% var)")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

        # Col 1 — by modality
        ax = axes[row, 1]
        for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
            m = all_modalities == mod
            ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
        ax.set_title(f"Level-0 {title_sfx}\nBy Modality")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    plt.suptitle("Level-0 Content vs Style Channel Subsets — PCA", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("latent_content_vs_style_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved latent_content_vs_style_pca.png")
else:
    print("Skipped — content_proj not active or all channels are content.")

## 8. Reconstruction visualisation

Compare original slices with their VQ-VAE-2 reconstructions.

## 9. Diagnostic label prediction

For each feature set (all encoder levels + level-0 content/style subsets) we train a
**logistic regression** and a **random forest** classifier on T1 features only (so modality
doesn't leak into the score), using stratified 5-fold cross-validation.

This tells us:
- **Which hierarchical level** encodes the most diagnostically-relevant information
- **Whether content or style channels** at level 0 carry the diagnostic signal
- How close any level is to chance (baseline = majority-class rate)

Results are shown as a bar chart and a summary table.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

# ── use T1 features only so modality can't trivially distinguish classes ──────
t1_mask = all_modalities == "T1"
y = all_labels[t1_mask]

# Build the feature sets to evaluate
feature_sets = {}
for i in range(nb_levels):
    feature_sets[f"Level {i} (all ch)"] = all_features[f"level_{i}"][t1_mask]

# Level-0 content / style subsets (only if content projection was active)
if all_content_indices is not None and len(all_style_indices) > 0:
    feature_sets["Level 0 — content"] = all_full_level0[t1_mask][:, all_content_indices]
    feature_sets["Level 0 — style"] = all_full_level0[t1_mask][:, all_style_indices]


# ── classifiers ───────────────────────────────────────────────────────────────
def make_lr():
    """Logistic Regression with standard scaling."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]
    )


def make_rf():
    """Random Forest (no scaling needed)."""
    return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=cv).mean()

print(f"{'Feature set':<28} {'LR acc':>8} {'LR ±':>7}  {'RF acc':>8} {'RF ±':>7}")
print("-" * 65)

results = []
for name, X in feature_sets.items():
    lr_scores = cross_val_score(make_lr(), X, y, cv=cv, scoring="accuracy")
    rf_scores = cross_val_score(make_rf(), X, y, cv=cv, scoring="accuracy")
    results.append(
        {
            "Feature set": name,
            "LR mean": lr_scores.mean(),
            "LR std": lr_scores.std(),
            "RF mean": rf_scores.mean(),
            "RF std": rf_scores.std(),
        }
    )
    print(
        f"{name:<28} {lr_scores.mean():.3f}   ±{lr_scores.std():.3f}   "
        f"{rf_scores.mean():.3f}   ±{rf_scores.std():.3f}"
    )

print("-" * 65)
print(f"{'Majority-class baseline':<28} {chance:.3f}")

results_df = pd.DataFrame(results)
print("\nSummary dataframe saved as  results_df  (use .to_csv() to export)")

In [ ]:
# ── Bar chart: LR and RF accuracy per feature set ────────────────────────────
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(results_df) * 1.6), 5))

bars_lr = ax.bar(
    x - width / 2,
    results_df["LR mean"],
    width,
    yerr=results_df["LR std"],
    capsize=4,
    label="Logistic Regression",
    color="steelblue",
    alpha=0.85,
)

bars_rf = ax.bar(
    x + width / 2,
    results_df["RF mean"],
    width,
    yerr=results_df["RF std"],
    capsize=4,
    label="Random Forest",
    color="tomato",
    alpha=0.85,
)

# Chance-level line
ax.axhline(chance, color="grey", linestyle="--", linewidth=1.2, label=f"Chance ({chance:.2f})")

# Annotate bars with their mean accuracy
for bar in list(bars_lr) + list(bars_rf):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results_df["Feature set"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("5-fold CV accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Diagnostic label prediction accuracy per VQ-VAE-2 feature set\n(T1 features only, 5-fold stratified CV)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("diagnostic_prediction_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved diagnostic_prediction_accuracy.png")

# ── Per-class breakdown (confusion matrix for best feature set) ──────────────
best_name = results_df.loc[results_df["RF mean"].idxmax(), "Feature set"]
best_X = feature_sets[best_name]
print(f"\nBest feature set by RF accuracy: '{best_name}'")
print("Showing confusion matrix (RF, 5-fold average) ...")

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred = cross_val_predict(make_rf(), best_X, y, cv=cv)
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[label_names[i] for i in sorted(label_names)])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix — RF on '{best_name}'")
plt.tight_layout()
plt.savefig("confusion_matrix_best_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrix_best_level.png")

print("\nClassification report:")
print(classification_report(y, y_pred, target_names=[label_names[i] for i in sorted(label_names)]))

In [ ]:
N_SHOW = 3  # number of subjects to show

fig, axes = plt.subplots(N_SHOW, 4, figsize=(14, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, item in enumerate(items[:N_SHOW]):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for col_offset, (key, mod_label) in enumerate([("image_t1", "T1"), ("image_t2", "T2")]):
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            recon, _, _, _, _, _ = vqvae_model(img, return_recon=True, pool_only=False)

        # Mid-sagittal slice (dim 2 = D)
        mid = img.shape[2] // 2
        orig_slice = img[0, 0, mid].cpu().numpy()
        recon_slice = recon[0, 0, mid].cpu().float().numpy()

        axes[row, col_offset * 2].imshow(orig_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2].set_title(f"Subject {row} — {mod_label} original")
        axes[row, col_offset * 2].axis("off")

        axes[row, col_offset * 2 + 1].imshow(recon_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2 + 1].set_title(f"Subject {row} — {mod_label} reconstruction")
        axes[row, col_offset * 2 + 1].axis("off")

plt.suptitle("VQ-VAE-2 Reconstructions (mid-sagittal)", fontsize=13)
plt.tight_layout()
plt.savefig("reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved reconstructions.png")

## 10. Save extracted features

In [ ]:
save_dict = {
    "labels": all_labels,
    "modalities": all_modalities,
    "label_map_keys": np.array(list(label_map.keys())),
    "label_map_values": np.array(list(label_map.values())),
    "checkpoint_step": np.array([checkpoint["step"]]),
    "content_indices": np.array(all_content_indices) if all_content_indices else np.array([]),
    "full_level0": all_full_level0,
}
for i in range(nb_levels):
    save_dict[f"features_level_{i}"] = all_features[f"level_{i}"]

np.savez("extracted_latents.npz", **save_dict)
print("✓ Saved extracted_latents.npz")
for i in range(nb_levels):
    print(f"  level_{i}: {all_features[f'level_{i}'].shape}")
print(f"  full_level0: {all_full_level0.shape}")